# 04 — Final selection, costs and statistical robustness

This notebook inspects the locked-test equity curves, transaction-cost sensitivity, bootstrap intervals and multiple-comparison-adjusted Diebold–Mariano tests.

In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = (ROOT / '..').resolve()
processed = ROOT / 'data/processed'
summary = json.loads((ROOT / 'reports/experiment_summary.json').read_text())
display(pd.DataFrame(summary['assets']).T[['winner', 'validation_score']])
gold_equity = pd.read_csv(processed / 'gold_test_equity.csv', index_col=0, parse_dates=True)
silver_equity = pd.read_csv(processed / 'silver_test_equity.csv', index_col=0, parse_dates=True)
gold_costs = pd.read_csv(processed / 'gold_cost_sensitivity.csv')
silver_costs = pd.read_csv(processed / 'silver_cost_sensitivity.csv')

,winner,validation_score
gold,extra_trees,0.4852
silver,hist_gradient_boosting,0.165137


In [2]:
sns.set_theme(style='whitegrid', context='notebook')
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
gold_equity['equity'].plot(ax=axes[0], color='#d49a00', title='Gold locked-test equity')
silver_equity['equity'].plot(ax=axes[0], color='#777777', title='Gold and Silver locked-test equity')
axes[0].set_ylabel('Growth of $1')
gold_equity['drawdown'].plot(ax=axes[1], color='#b33a3a', label='Gold')
silver_equity['drawdown'].plot(ax=axes[1], color='#555555', label='Silver')
axes[1].set_title('Drawdown')
axes[1].legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(gold_costs['transaction_cost_bps'], gold_costs['sharpe'], marker='o', label='Gold')
ax.plot(silver_costs['transaction_cost_bps'], silver_costs['sharpe'], marker='o', label='Silver')
ax.axhline(0, color='black', linewidth=0.8)
ax.set(title='Sharpe sensitivity to transaction costs', xlabel='Cost (bps per turnover unit)', ylabel='Net Sharpe')
ax.legend()
plt.show()

/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_17889/1475481936.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_17889/1475481936.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
for asset in ['gold', 'silver']:
    tests = pd.read_csv(processed / f'{asset}_statistical_tests.csv')
    print(asset.upper())
    display(tests[['competitor', 'dm_statistic', 'p_value_holm', 'significant_5pct_holm', 'sharpe_difference', 'difference_ci_low', 'difference_ci_high']])

gold_tests = pd.read_csv(processed / 'gold_statistical_tests.csv')
plot_tests = gold_tests.copy()
plot_tests['minus_log10_holm_p'] = (-plot_tests['p_value_holm'].clip(lower=1e-300).apply(lambda x: __import__('math').log10(x)))
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_tests.sort_values('minus_log10_holm_p'), x='competitor', y='minus_log10_holm_p', color='#244a7c')
plt.axhline(-__import__('math').log10(0.05), color='#b33a3a', linestyle='--', label='5% threshold')
plt.xticks(rotation=35, ha='right')
plt.title('Gold: evidence against equal forecast accuracy after Holm correction')
plt.legend()
plt.show()

GOLD


,competitor,dm_statistic,p_value_holm,significant_5pct_holm,sharpe_difference,difference_ci_low,difference_ci_high
0,zero,-1.482273,6.913378e-01,False,0.779326,-0.192650,1.650535
1,elasticnet,-0.948930,9.781442e-01,False,1.207878,0.191432,2.460190
2,hist_gradient_boosting,-0.982105,9.781442e-01,False,0.303436,-0.698053,1.308424
3,moving_average,-3.101511,1.347748e-02,True,0.514700,-0.771589,1.795362
4,ridge,-0.393288,9.781442e-01,False,0.778267,-0.520321,2.031328
5,timesfm,-1.250689,8.441926e-01,False,0.504786,-0.664481,1.535330
6,tsmixer,-16.163118,9.179895e-58,True,1.389345,0.006454,2.645221
7,time_mixer,-11.241623,2.291913e-28,True,1.916810,0.738005,3.223884
8,last_return,-8.908904,4.123151e-18,True,2.480405,1.301584,3.665711
9,chronos,-1.703946,5.303474e-01,False,1.188533,0.036436,2.343454


SILVER


,competitor,dm_statistic,p_value_holm,significant_5pct_holm,sharpe_difference,difference_ci_low,difference_ci_high
0,zero,0.295377,1.000000e+00,False,0.025078,-0.939637,0.783979
1,ridge,1.335706,9.082269e-01,False,-0.371079,-1.368429,0.681536
2,elasticnet,0.995375,1.000000e+00,False,-0.255673,-1.067418,0.644146
3,moving_average,-2.416962,1.095547e-01,False,-0.125727,-1.607944,1.096750
4,extra_trees,2.030170,2.540354e-01,False,-0.833756,-1.653564,-0.173341
5,time_mixer,-9.773957,1.310850e-21,True,0.515043,-0.687104,1.696014
6,timesfm,-0.964501,1.000000e+00,False,0.263321,-1.139454,1.415993
7,patch_tst,-20.864990,1.225592e-95,True,0.870797,-0.717415,2.163361
8,tsmixer,-15.899643,6.372669e-56,True,0.250550,-1.309552,1.485929
9,last_return,-4.651349,2.638165e-05,True,0.837231,-0.526097,1.989675


/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_17889/1959690510.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
